# Приложение Д. Эффективная настройка параметров с помощью LoRA

In [1]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow", # Для предобученных весов OpenAI
        "pandas"      # Загрузка наборов данных
       ]
for p in pkgs:
    print(f"{p} версия: {version(p)}")

matplotlib версия: 3.10.9
numpy версия: 2.4.6
tiktoken версия: 0.13.0
torch версия: 2.12.0
tensorflow версия: 2.21.0
pandas версия: 3.0.3


## Д.1 Введение в LoRA

- Низкоранговая адаптация (Low-rank adaptation (LoRA)) — это метод машинного обучения, который модифицирует предобученную модель для лучшего соответствия конкретному, часто меньшему набору данных путём настройки лишь небольшого низкорангового подмножества параметров модели
- Этот подход важен, поскольку он позволяет эффективно дообучать большие модели на данных, специфичных для конкретной задачи, значительно снижая вычислительные затраты и время, необходимые для тонкой настройки

- Предположим, у нас есть большая весовая матрица $W$ для заданного слоя
- Во время обратного распространения мы изучаем матрицу $\Delta W$, которая содержит информацию о том, насколько мы хотим обновить исходные веса, чтобы минимизировать функцию потерь в процессе обучения
- При обычном обучении и тонкой настройке обновление весов определяется следующим образом:

$$W_{\text{updated}} = W + \Delta W$$

- Метод LoRA, предложенный [Hu et al.](https://arxiv.org/abs/2106.09685), предлагает более эффективную альтернативу вычислению обновлений весов $\Delta W$ путём изучения его аппроксимации, $\Delta W \approx AB$
- Другими словами, в LoRA мы имеем следующее, где $A$ и $B$ — две малые весовые матрицы:

$$W_{\text{updated}} = W + AB$$

- Рисунок ниже иллюстрирует эти формулы для полной тонкой настройки и LoRA рядом друг с другом

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-e_compressed/lora-1.webp" width="800px">

- Изображения полной тонкой настройки и LoRA на рисунке выше выглядят немного иначе, чем формулы, которые были приведены ранее
- Это связано с дистрибутивным законом умножения матриц: нам не нужно складывать веса с обновлёнными весами, а можно держать их раздельно
- Например, если $x$ — это входные данные, то для обычной тонкой настройки мы можем записать следующее:

$$x (W+\Delta W) = x W + x \Delta W$$

- Аналогично, для LoRA мы можем записать следующее:

$$x (W+A B) = x W + x A B$$

- Тот факт, что мы можем держать весовые матрицы LoRA отдельно, делает LoRA особенно привлекательной
- На практике это означает, что нам вообще не нужно изменять веса предобученной модели, так как мы можем применять матрицы LoRA на лету
- После настройки набора данных и загрузки модели мы реализуем LoRA в коде, чтобы сделать эти концепции менее абстрактными

## Д.2 Подготовка набора данных

In [2]:
import requests
from pathlib import Path
import pandas as pd
from previous_chapters import (
    download_and_unzip_spam_data,
    create_balanced_dataset,
    random_split
)


url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"Основной URL не сработал: {e}. Пробуем резервный URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
balanced_df = create_balanced_dataset(df)
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

Файл скачан и сохранён как sms_spam_collection\SMSSpamCollection.tsv


In [3]:
import torch
import tiktoken
from previous_chapters import SpamDataset


tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset("train.csv", max_length=None, tokenizer=tokenizer)
val_dataset = SpamDataset("validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
test_dataset = SpamDataset("test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)

In [4]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

- В качестве проверочного шага мы итерируем по загрузчикам данных и проверяем, что каждый пакет содержит по 8 обучающих примеров, где каждый обучающий пример состоит из 120 токенов

In [5]:
print("Загрузчик обучающих данных:")
for input_batch, target_batch in train_loader:
    pass

print("Размеры входного пакета:", input_batch.shape)
print("Размеры пакета меток:", target_batch.shape)

Загрузчик обучающих данных:
Размеры входного пакета: torch.Size([8, 120])
Размеры пакета меток: torch.Size([8])


- Выведем общее количество пакетов в каждом наборе данных

In [6]:
print(f"{len(train_loader)} обучающих пакетов")
print(f"{len(val_loader)} валидационных пакетов")
print(f"{len(test_loader)} тестовых пакетов")

130 обучающих пакетов
19 валидационных пакетов
38 тестовых пакетов


## Д.3 Инициализация модели

In [7]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt


CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # Размер словаря
    "context_length": 1024,  # Длина контекста
    "drop_rate": 0.0,        # Коэффициент дропаута
    "qkv_bias": True         # Смещение для query-key-value
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 77.2kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 1.10MiB/s]
hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 90.0kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [04:14<00:00, 1.96MiB/s]  
model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 5.20MiB/s]
model.ckpt.meta: 100%|██████████| 471k/471k [00:00<00:00, 581kiB/s] 
vocab.bpe: 100%|██████████| 456k/456k [00:00<00:00, 676kiB/s]  


- Чтобы убедиться, что модель загружена корректно, давайте дважды проверим, генерирует ли она связный текст

In [8]:
from previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)


text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work


- Затем мы подготавливаем модель к тонкой настройке для классификации, где мы заменяем выходной слой.

In [9]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=768, out_features=num_classes)

In [10]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Устройство:", device)

model.to(device);  # для классов nn.Module присваивание model = model.to(device) не требуется

Устройство: cpu


- Наконец, давайте вычислим начальную точность классификации недонастроенной модели (мы ожидаем, что она будет около 50%, что означает, что модель пока не способна надёжно различать спам- и не спам-сообщения)

In [11]:
from previous_chapters import calc_accuracy_loader

torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Точность на обучающем наборе: {train_accuracy*100:.2f}%")
print(f"Точность на валидационном наборе: {val_accuracy*100:.2f}%")
print(f"Точность на тестовом наборе: {test_accuracy*100:.2f}%")

Точность на обучающем наборе: 46.25%
Точность на валидационном наборе: 45.00%
Точность на тестовом наборе: 48.75%
